# JET-MLIPs with ASE

Run this notebook in the env of one family (README sections 1-2: the official package, then JET).
Set `FAMILY` below; every other cell works for all five families.

In [ ]:
FAMILY = 'mace'   # 'sevenn', 'mace', 'nequip', 'nequix' or 'equflash'
EQUFLASH_CHECKPOINT = 'equflash_oam_30M.pt'   # EquFlash only: path of the checkpoint file

## 1. The calculator

Each family keeps its official calculator; JET replaces the model's layers in place.

In [ ]:
def make_calc(family):
    if family == 'sevenn':
        from sevenn.calculator import SevenNetCalculator
        return SevenNetCalculator('7net-0', enable_jet=True)
    if family == 'mace':
        from mace.calculators import mace_mp
        from jet_mace import convert
        calc = mace_mp(model='medium', device='cuda', default_dtype='float32')
        calc.models[0] = convert.patch_model(calc.models[0], cueq=True)
        return calc
    if family == 'nequip':
        from nequip.integrations.ase import NequIPCalculator
        from jet_nequip import convert
        calc = NequIPCalculator._from_saved_model('nequip.net:mir-group/NequIP-OAM-M:0.1', device='cuda',
                                                  chemical_species_to_atom_type_map={'Si': 'Si'})   # map: your elements
        calc.model = convert.patch_model(calc.model)
        return calc
    if family == 'nequix':
        from nequix.calculator import NequixCalculator
        from jet_nequix import convert
        calc = NequixCalculator('nequix-mp-1', backend='torch', use_kernel=False)
        calc.model = convert.patch_model(calc.model)
        return calc
    if family == 'equflash':
        from GGNN.common.calculator import UCalculator
        from jet_equflash import convert
        calc = UCalculator(checkpoint_path=EQUFLASH_CHECKPOINT, cpu=False)
        convert.patch_model(getattr(calc.trainer.model, 'module', calc.trainer.model))
        return calc
    raise ValueError(family)

calc = make_calc(FAMILY)

## 2. Energy, forces, stress

In [ ]:
import time
from ase.build import bulk

atoms = bulk('Si', 'diamond', a=5.43, cubic=True).repeat((4, 4, 4))   # 512 atoms
atoms.calc = calc
print('energy (eV)', atoms.get_potential_energy())
print('max |force| (eV/A)', abs(atoms.get_forces()).max())
print('stress (eV/A^3)', atoms.get_stress())

atoms.rattle(0.01)   # new positions: the next call recomputes
t = time.perf_counter(); atoms.get_forces(); print(f'one call: {(time.perf_counter() - t) * 1e3:.1f} ms')

## 3. Short MD

In [ ]:
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

MaxwellBoltzmannDistribution(atoms, temperature_K=300)
md = Langevin(atoms, 1.0 * units.fs, temperature_K=300, friction=0.01 / units.fs)
md.attach(lambda: print(f'step {md.nsteps:4d}  T {atoms.get_temperature():6.1f} K  '
                        f'E_pot {atoms.get_potential_energy():.4f} eV'), interval=20)
t = time.perf_counter(); md.run(100)
print(f'{(time.perf_counter() - t) / 100 * 1e3:.1f} ms/step')

## 4. D3 dispersion

`JetD3Calculator` is the GPU kernel of the LAMMPS `jet/d3` pair style. Build it once per env, from the repository folder:

```bash
JET_CUDA_ARCH=90 pip install -e jet_d3/ase --no-build-isolation
```

`xc` is the functional of the model's training data (e.g. `'pbe'`, `'r2scan'`); `damping='bj'` (default) or `'zero'`.
On a consumer GeForce card, set `JET_D3_FP64=0` in the environment before the calculator is created (README, D3).

In [ ]:
from ase.calculators.mixing import SumCalculator
from jet_d3_ase import JetD3Calculator

atoms.calc = SumCalculator([calc, JetD3Calculator(xc='pbe')])   # model + D3(BJ, PBE)
print('energy with D3 (eV)', atoms.get_potential_energy())